In [181]:
import copy
import os
import argparse
from erasure.utils.logger import GLogger
import torch
import numpy as np
import random
from erasure.utils.config.local_ctx import Local
from erasure.utils.config.global_ctx import Global, bcolors 
from erasure.core.factory_base import ConfigurableFactory
from erasure.data.datasets.DatasetManager import DatasetManager

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [182]:
config_file = os.path.join("configs","LinkAttack", "Citeseer_edge.jsonc")

In [183]:
global_ctx = Global(config_file)
global_ctx.factory = ConfigurableFactory(global_ctx)

#Create Dataset
data_manager = global_ctx.factory.get_object( Local( global_ctx.config.data ))
global_ctx.dataset = data_manager

@,-710922381 | INFO | 3846698 - Creating Global Context for: configs/LinkAttack/Citeseer_edge.jsonc


0m���,-710922367 | INFO | 3846698 - Setting seeds to: 0
!,-710922335 | INFO | 3846698 - REMOVAL TYPE SET AS edge
,-710922334 | INFO | 3846698 - Caching System: False.
�	@��,-710922305 | INFO | 3846698 - Instantiating: torch_geometric.datasets.Planetoid
Ќ�n�,-710922152 | INFO | 3846698 - Created Configurable: erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource
PXH[�,-710922151 | INFO | 3846698 - {'class': 'erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource', 'parameters': {'datasource': {'class': 'torch_geometric.datasets.Planetoid', 'parameters': {'root': 'resources/data', 'name': 'Citeseer'}, 'preprocess': []}}}
�	@��,-710922115 | INFO | 3846698 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
��3	�,-710922065 | INFO | 3846698 - ['all', 'all_shuffled', '-']
�	@��,-710922064 | INFO | 3846698 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
�lVX�,-710922055 | INFO | 38

In [184]:
#Create Predictor
current = Local(global_ctx.config.predictor)
current.dataset = data_manager
predictor = global_ctx.factory.get_object(current)
global_ctx.predictor = predictor
global_ctx.logger.info('Global Predictor: ' + str(predictor))

�	@��,-710921893 | INFO | 3846698 - Instantiating: erasure.model.graphs.GCN.GCN


�	@��,-710921885 | INFO | 3846698 - Instantiating: torch.optim.Adam
�	@��,-710921884 | INFO | 3846698 - Instantiating: torch.nn.CrossEntropyLoss
graph has edges  torch.Size([2, 9104])
��t,-710921814 | INFO | 3846698 - epoch = 0 ---> loss = 1.7978	 accuracy = 0.1720
graph has edges  torch.Size([2, 9104])
��t,-710921748 | INFO | 3846698 - epoch = 1 ---> loss = 1.7470	 accuracy = 0.3599
graph has edges  torch.Size([2, 9104])
��t,-710921689 | INFO | 3846698 - epoch = 2 ---> loss = 1.6973	 accuracy = 0.5436
graph has edges  torch.Size([2, 9104])
��t,-710921617 | INFO | 3846698 - epoch = 3 ---> loss = 1.6504	 accuracy = 0.6200
graph has edges  torch.Size([2, 9104])
��t,-710921564 | INFO | 3846698 - epoch = 4 ---> loss = 1.5992	 accuracy = 0.6785
graph has edges  torch.Size([2, 9104])
��t,-710921508 | INFO | 3846698 - epoch = 5 ---> loss = 1.5530	 accuracy = 0.7027
graph has edges  torch.Size([2, 9104])
��t,-710921443 | INFO | 3846698 - epoch = 6 ---> loss = 1.5004	 accuracy = 0.7219
graph 

In [185]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [186]:
data_manager.partitions['all'][0][1]

tensor([3, 1, 5,  ..., 3, 1, 5])

In [187]:
import torch
print(torch.version.cuda)

12.6


In [188]:
print(predictor.optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.001
    lr: 0.0005000000000000002
    maximize: False
    weight_decay: 0
)


In [189]:
import torch
print(torch.__version__)
print(torch.version.cuda)


2.7.0+cu126
12.6


In [190]:
#Create unlearners 
unlearners = []
unlearners_cfg = global_ctx.config.unlearners
for un in unlearners_cfg:
    current = Local(un)
    current.dataset = data_manager
    current.predictor = copy.deepcopy(predictor)
    unlearners.append( global_ctx.factory.get_object(current) )

�	@��,-710918590 | INFO | 3846698 - Instantiating: torch.optim.Adam
Ќ�n�,-710918588 | INFO | 3846698 - Created Configurable: erasure.unlearners.graph_unlearners.Finetuning.Finetuning
�	@��,-710918552 | INFO | 3846698 - Instantiating: torch.optim.Adam
Ќ�n�,-710918551 | INFO | 3846698 - Created Configurable: erasure.unlearners.graph_unlearners.NegGrad.NegGrad
�	@��,-710918489 | INFO | 3846698 - Instantiating: torch.optim.SGD
Ќ�n�,-710918488 | INFO | 3846698 - Created Configurable: erasure.unlearners.graph_unlearners.AdvancedNegGrad.AdvancedNegGrad
�	@��,-710918427 | INFO | 3846698 - Instantiating: torch.optim.SGD
Ќ�n�,-710918399 | INFO | 3846698 - Created Configurable: erasure.unlearners.graph_unlearners.BadTeaching.BadTeaching


/NFSHOME/adangelo/LinkInference/erasure/unlearners/graph_unlearners/GraphUnlearner.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = torch.tensor(self.labels)


In [191]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [192]:
data_manager.partitions['all'][0][0].edge_index

tensor([[   0,    1,    1,  ..., 3324, 3325, 3326],
        [ 628,  158,  486,  ..., 2820, 1643,   33]])

In [193]:
data_manager.partitions['all'].revise_graph_edges([(0,1),(0,2)])[0][0]

Data(x=[3327, 3703], edge_index=[2, 0])

In [194]:
print(len( data_manager.partitions['forget']) )

1820


In [195]:
#Evaluator
current = Local(global_ctx.config.evaluator)
current.unlearners = unlearners
evaluator = global_ctx.factory.get_object(current)

# Evaluations
for unlearner in unlearners:
    global_ctx.logger.info(f'''{bcolors.OKGREEN}####\t\t Evaluating Unlearner {unlearner.__class__.__name__} \t\t####{bcolors.ENDC}''')
    evaluator.evaluate(unlearner,predictor)

Ќ�n�,-710917985 | INFO | 3846698 - Created Configurable: erasure.evaluations.running.RunTime
��QX�,-710917983 | INFO | 3846698 - Function: <function accuracy_score at 0x7fd15b5ae3a0>
Ќ�n�,-710917982 | INFO | 3846698 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
p�QX�,-710917961 | INFO | 3846698 - Function: <function accuracy_score at 0x7fd15b5ae3a0>
Ќ�n�,-710917960 | INFO | 3846698 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
@�3X�,-710917930 | INFO | 3846698 - Function: <function accuracy_score at 0x7fd15b5ae3a0>
Ќ�n�,-710917929 | INFO | 3846698 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
,-710917898 | INFO | 3846698 - Function: <function accuracy_score at 0x7fd15b5ae3a0>
Ќ�n�,-710917898 | INFO | 3846698 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
Ќ�n�,-710917897 | INFO | 3846698 - Created Configurable: erasure.evaluations.measures.SaveValues
Ќ�n�,-710917897 | INFO | 3846

,-710917487 | INFO | 3846698 - Unlearning copyed predictor: <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7fd1580cf520>
��E	�,-710917486 | INFO | 3846698 - Starting NegGrad with 1 epochs
,-710917123 | INFO | 3846698 - NegGrad - epoch = 0 ---> var_loss = -0.5587
,-710917076 | INFO | 3846698 - sklearn.metrics.accuracy_score on partition: "test", target model: unlearned, unlearned graph: True: 0.7492492492492493 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7fd1580cf520>
,-710917040 | INFO | 3846698 - sklearn.metrics.accuracy_score on partition: "test", target model: original, unlearned graph: True: 0.7612612612612613 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7fd150941d00>
,-710917004 | INFO | 3846698 - sklearn.metrics.accuracy_score on partition: "test", target model: unlearned, unlearned graph: False: 0.7732732732732732 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7fd1580cf520>
,-710916968 | INFO | 3846698 - sklearn.metr